<div style="width: 70%;">
    <img src="../ONS_Logo_Digital_Colour_Landscape_English_RGB.svg" alt="ONS Logo">
</div>


# ClassifAI Demo

This demo uses a mock occupations dataset to show how ClassifAI matches unlabelled job descriptions to SOC codes using an existing knowledgebase.

In [ ]:
import re

from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput
from classifai.indexers.hooks import (
    CapitalisationStandardisingHook,
    DeduplicationHook,
)
from classifai.indexers.hooks.hook_factory import HookBase

from demo_utils import (
    KNOWLEDGEBASE_PATH,
    load_uncoded_input,
    load_vectoriser,
    make_query_input,
    prepare_knowledgebase,
)

prepare_knowledgebase()
uncoded_input = load_uncoded_input()
vectoriser = load_vectoriser()


basic_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
)

### Example Responses

The examples below use a small batch of uncoded occupation descriptions. In a real workflow, these could come from survey responses, form submissions, or operational data waiting to be coded.

---
## 1. Return A Best Match

At the simplest level, you pass in uncoded text and get back the highest-ranked suggestion for each response.

In [2]:
basic_vectorstore.search(query=uncoded_input, n_results=1)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 74.77it/s]


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008
1,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646
2,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518


---
## 2. Return A Shortlist For Review

When a single suggestion is not enough, the same search can return a ranked shortlist for review.

In [3]:
basic_vectorstore.search(query=uncoded_input, n_results=3)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 89.56it/s]


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Fruit farmer: Grows and harvests fruits such a...,2,0.820159
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,3,0.782137
3,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646
4,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Vegetable farmer: Cultivates and harvests vege...,2,0.738995
5,1,Cow Farmer: Manages dairy and beef cattle oper...,101,Fruit farmer: Grows and harvests fruits such a...,3,0.737185
6,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518
7,2,Machine Learning Engineer: Designs and deploys...,107,Web developer: Builds and maintains websites a...,2,0.681587
8,2,Machine Learning Engineer: Designs and deploys...,104,"Carpenter: Constructs, installs, and repairs w...",3,0.641518


## 3. Return Extra Context With Each Match

The matched knowledgebase text is useful on its own, but teams often need more context alongside each suggestion.

Extra columns such as sector can support analyst review, help group similar results, and make it easier to route suggestions into downstream workflows.

In [51]:
metadata_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    meta_data={"sector": str},
)

metadata_vectorstore.search(query=uncoded_input, n_results=1)

,query_id,query_text,doc_label,doc_text,rank,score,sector
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.844008,Agriculture
1,1,Cow Farmer: Manages dairy and beef cattle oper...,102,Dairy farmer: Manages cows for milk production...,1,0.894646,Agriculture
2,2,Machine Learning Engineer: Designs and deploys...,107,"Software developer: Designs, writes, and tests...",1,0.737518,Technology


---
## 4. Use Hooks To Handle Messy Input

Hooks let you adapt behaviour without rewriting the search call.

Here the input is deliberately inconsistent. `CapitalisationStandardisingHook` normalises the text before embedding. This is useful as data from different sources may having varying cases.

In [52]:
messy_input = make_query_input([
    "TOMATO FARMER",
    "Machine Learning ENGINEER",
    "pHd StUdEnT",
])

hook_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={"search_preprocess": CapitalisationStandardisingHook(method="lower")},
)

hook_vectorstore.search(query=messy_input, n_results=1)

,query_id,query_text,doc_label,doc_text,rank,score
0,0,tomato farmer,101,Vegetable farmer: Cultivates and harvests vege...,1,0.769499
1,1,machine learning engineer,107,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,phd student,107,"Software developer: Designs, writes, and tests...",1,0.627140


### Keep Shortlists Clean

If several knowledgebase entries share the same label, the raw shortlist can contain repeated labels. `DeduplicationHook` trims that down to one best match per label, which makes review screens easier to work through.

In [49]:
dedup_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={"search_postprocess": DeduplicationHook(score_aggregation_method="max")},
)

review_query = make_query_input([
    "Tomato Farmer: Cultivates and harvests tomatoes in large greenhouse operations."
])

raw_shortlist = basic_vectorstore.search(query=review_query, n_results=5)
clean_shortlist = dedup_vectorstore.search(query=review_query, n_results=5)

print("\nRaw shortlist")
display(raw_shortlist)

print("After deduplication")
display(clean_shortlist)

Processing query batches: 100%|██████████| 1/1 [00:00<00:00, 272.55it/s]


Raw shortlist


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Fruit farmer: Grows and harvests fruits such a...,2,0.801251
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,3,0.791010
3,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,"Sheep farmer: Raises sheep for wool, meat, and...",4,0.755043
4,0,Tomato Farmer: Cultivates and harvests tomatoe...,104,"Carpenter: Constructs, installs, and repairs w...",5,0.661157


After deduplication


,query_id,query_text,doc_label,doc_text,rank,score
0,0,Tomato Farmer: Cultivates and harvests tomatoe...,101,Vegetable farmer: Cultivates and harvests vege...,1,0.816154
1,0,Tomato Farmer: Cultivates and harvests tomatoe...,102,Dairy farmer: Manages cows for milk production...,2,0.791010
2,0,Tomato Farmer: Cultivates and harvests tomatoe...,104,"Carpenter: Constructs, installs, and repairs w...",3,0.661157


---
## 5. Add A Small Project-Specific Hook

Built-in hooks cover common cases, but project teams often have their own abbreviations and shorthand.

Custom hooks can solve these sorts of problems in a concise and reusable way.

In [ ]:
class AbbreviationExpansionHook(HookBase):
    def __init__(self, colname: str = "query"):
        super().__init__(colname=colname, hook_type="pre_processing")
        self.colname = colname

    def _expand_query(self, text: str) -> str:
        text = text.lower()
        text = re.sub(r"\bml\b", "machine learning", text)
        text = re.sub(r"\bdev\b", "developer", text)
        text = re.sub(r"\bons\b", "office for national statistics", text)
        return text

    def __call__(self, input_data: VectorStoreSearchInput) -> VectorStoreSearchInput:
        if self.colname not in input_data.columns:
            raise ValueError(f"Column {self.colname!r} not found in input data.")

        processed_input = input_data.copy()
        processed_input[self.colname] = (
            processed_input[self.colname].astype(str).apply(self._expand_query)
        )
        return input_data.__class__.validate(processed_input)


custom_vectorstore = VectorStore(
    file_name=str(KNOWLEDGEBASE_PATH),
    data_type="csv",
    vectoriser=vectoriser,
    skip_save=True,
    quiet_mode=True,
    hooks={"search_preprocess": AbbreviationExpansionHook()},
)

custom_input = make_query_input(["dev", "ml engineer", "ons employee"])

custom_vectorstore.search(query=custom_input, n_results=1)

,query_id,query_text,doc_label,doc_text,rank,score
0,0,developer,107,Web developer: Builds and maintains websites a...,1,0.802484
1,1,machine learning engineer,107,"Software developer: Designs, writes, and tests...",1,0.702236
2,2,office for national statistics employee,102,Dairy farmer: Manages cows for milk production...,1,0.576905
